# Record3D Point Cloud Visualization

This notebook visualizes combined point clouds from all cameras to check alignment.


In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
!source /workspace/Home_Reconstruction/venv/bin/activate
!pip install --upgrade ipykernel
!/workspace/Home_Reconstruction/venv/bin/python -m ipykernel install --user --name=home_recon_venv --display-name "Python (home_recon_venv)"
!pip install opencv-python
!pip install scikit-image scikit-learn
!pip install ninja


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Installed kernelspec home_recon_venv in /root/.local/share/jupyter/kernels/home_recon_venv

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
%autoreload 2
import sys
print(sys.executable)

import OpenEXR
from pathlib import Path

PROJECT_ROOT = Path('/workspace/Home_Reconstruction')
sys.path.insert(0, str(PROJECT_ROOT))  # only the parent folder

import scene
from scene.objectgs_model import *
from scene.train import *
from scene.vis_tools.pc_viewer import *

import scene.data_loaders 
from scene.data_loaders.record3d_loader import *
from scene.data_loaders.generate_masks import *

import gsplat

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
%matplotlib inline

/workspace/Home_Reconstruction/venv/bin/python
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Load And View Scene

In [2]:
%autoreload 2
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ['PYTHONIOENCODING'] = 'utf-8'
from scene.data_loaders.generate_masks import *
from huggingface_hub import login
#paste login command w token here
scene_dir = "/workspace/Home_Reconstruction/data_scenes/maria_bedroom"
text_prompts = ["chair","book", "guitar", "bed", "plant", "lamp", "wall", "dresser", "mirror", "shelf", "picture", "postcard", "bottle", "computer", "clothing", "desk", "record player", "computer", "desk", "paper"]
n=20

# masks_dir, class_mapping = segment_and_save_incremental(
#     text_prompts=text_prompts,
#     scene_path=scene_dir,
#     save_vis=True,
#     n=(n*4),
#     resume=False,
    #)

In [3]:
# Step 3: Use the class via the module (no wildcard import)
%autoreload 2
scene_obj = Record3DScene(
    scene_path=scene_dir,
    use_semantics=True,
    subsample=4,
    frame_step=n,
    redo_semantics=False,
)

# Step 4: Check results
print(f"\nSemantic Point Cloud Summary:")
print(f"  Total points: {len(scene_obj.points):,}")
print(f"  Unique objects: {scene_obj.num_objects}")
print(f"  Object IDs: {scene_obj.get_object_ids_list()}")

Loaded metadata from /workspace/Home_Reconstruction/data_scenes/maria_bedroom/EXR_RGBD/metadata.json
  Image dimensions: 720x960
  Number of frames: 9720

Dataset split:
  Training: 486 frames
  Testing:  97 frames

Creating 486 cameras...


Loading cameras: 100%|██████████| 486/486 [00:25<00:00, 19.39it/s]


Created 486 cameras

Creating 97 cameras...


Loading cameras: 100%|██████████| 97/97 [00:04<00:00, 20.79it/s]


Created 97 cameras

Loading point cloud...
Loading semantic point cloud from /workspace/Home_Reconstruction/data_scenes/maria_bedroom/processed_semantic.ply
  Loaded 1,877,786 points with 78 unique objects

Semantic Point Cloud Summary:
  Total points: 1,877,786
  Unique objects: 78
  Object IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77]


In [6]:
%autoreload 2
# Visualize with semantic colors and legend
visualize_pointclouds(
    scene_dir,
    ["processed_semantic.ply"],
    color_by_semantics=True,
    prompts=text_prompts,
)
# # Visualize with semantic colors and legend
# visualize_pointclouds(
#     scene_dir,
#     ["processed_semantic.ply"],
#     color_by_semantics=False,
#     prompts=text_prompts,
# )

✓ Loaded remapped class mapping: 78 objects

Loading 1 point cloud(s) from maria_bedroom
🎨 PANOPTIC coloring: class color + instance opacity
   Background opacity: 10%
   Instance opacity range: 40% - 100%

Loading processed_semantic.ply...
  ✓ Loaded object IDs: 78 unique objects

  Classes (18):
    bed: 2 instance(s), 59,203 pts
    book: 4 instance(s), 3,873 pts
    bottle: 10 instance(s), 3,847 pts
    chair: 2 instance(s), 24,218 pts
    clothing: 18 instance(s), 62,768 pts
    computer: 2 instance(s), 16,908 pts
    desk: 1 instance(s), 10,055 pts
    dresser: 1 instance(s), 39,170 pts
    guitar: 2 instance(s), 9,401 pts
    lamp: 1 instance(s), 6,145 pts
    mirror: 2 instance(s), 12,671 pts
    paper: 4 instance(s), 4,859 pts
    picture: 9 instance(s), 16,291 pts
    plant: 2 instance(s), 899 pts
    postcard: 4 instance(s), 2,233 pts
    record player: 1 instance(s), 4,026 pts
    shelf: 4 instance(s), 13,065 pts
    wall: 8 instance(s), 682,946 pts
  ✓ 1,877,786 points tot


Summary:
processed_semantic.ply          1,877,786 points (150,000 displayed)

🎨 Class Legend (18 classes + background):
------------------------------------------------------------
  background           color=(0.50,0.50,0.50)  opacity=10%
  bed                  color=(0.85,0.60,0.21)   2 inst    59,203 pts  opacity=40%-100%
  book                 color=(0.40,0.16,0.95)   4 inst     3,873 pts  opacity=40%-100%
  bottle               color=(0.08,0.85,0.09)  10 inst     3,847 pts  opacity=40%-100%
  chair                color=(0.95,0.24,0.43)   2 inst    24,218 pts  opacity=40%-100%
  clothing             color=(0.14,0.54,0.85)  18 inst    62,768 pts  opacity=40%-100%
  computer             color=(0.83,0.95,0.09)   2 inst    16,908 pts  opacity=40%-100%
  desk                 color=(0.75,0.21,0.85)   1 inst    10,055 pts  opacity=100%
  dresser              color=(0.16,0.95,0.60)   1 inst    39,170 pts  opacity=100%
  guitar               color=(0.85,0.28,0.08)   2 inst     9,401 pts  

In [9]:
# TRAIN A MODEL FROM SCRATCH
import os
os.environ["PYTHONIOENCODING"] = "utf-8"

import torch
import importlib
import glob
from pathlib import Path

# Reload
import scene.train
import scene.trainer_vcd
import scene.objectgs_model
import scene.objectgs_model_vcd

importlib.reload(scene.train)
importlib.reload(scene.objectgs_model)
importlib.reload(scene.trainer_vcd)
importlib.reload(scene.objectgs_model_vcd)

from scene.objectgs_model_vcd import ObjectGSModelVCD
from scene.trainer_vcd import GaussianTrainerVCD

from scene.train import GaussianTrainer
from scene.objectgs_model import ObjectGSModel

RESUME_FROM_CHECKPOINT = True
training_run_n = "training_run_9"

model = ObjectGSModelVCD(
    point_cloud=scene_obj.points,
    colors=scene_obj.colors,
    object_ids=scene_obj.object_ids,
    object_names=text_prompts,
    voxel_size=0.02,
    k=10,
).to("cuda")

config = {
    # =================================================================
    # LEARNING RATES  (unchanged — already validated)
    # =================================================================
    "lr": 0.001,
    "lr_feature": 0.0025,
    "lr_position": 0.00016,
    "lr_scaling": 0.005,

    # =================================================================
    # TRAINING DURATION
    # =================================================================
    "num_iterations": 10000,
    "eval_interval": 1000,

    # =================================================================
    # PROGRESSIVE RESOLUTION
    # =================================================================
    "use_progressive_resolution": True,
    "progressive_resolution_schedule": [
        (0.02, 4),
        (0.08, 2),
        (0.90, 1),
    ],

    # =================================================================
    # LOSS WEIGHTS
    # =================================================================
    "lambda_ssim": 0.3,
    "lambda_volume": 0.0001,
    "lambda_scale_reg": 8.0,
    "max_scale": 0.02,
    "lambda_offset_leash": 0.3,
    "adaptive_offset_cap_multiplier": 3.0,

    # =================================================================
    # SEMANTIC LOSS
    # =================================================================
    "use_semantic_loss": True,
    "lambda_semantic": 0.05,
    "semantic_loss_start": 4000,
    "semantic_warmup_iters": 1500,

    # =================================================================
    # DENSIFICATION / PRUNING
    # =================================================================
    "use_densification": True,

    "densify_start": 100,
    "densify_until": 9000,
    "densify_interval": 500,

    "densify_grad_threshold": 1e-5,
    "prune_opacity_threshold": 0.005,

    # NEW — required by updated pruning logic
    "prune_warmup_iters": 2000,     # iteration-based warmup
    "prune_grad_factor": 0.25,      # gradient must be < 0.25 * densify threshold
    "min_cycles": 5,                # sustained low-gradient cycles (500 * 5 = 2500 iters)

    # =================================================================
    # CHECKPOINTING
    # =================================================================
    "checkpoint_topk": 3,
    "checkpoint_metric": "test_l1",
    "lower_is_better": True,
    "progress_render_scale": 0.5,

    # =================================================================
    # EARLY STOPPING
    # =================================================================
    "early_stop_patience_evals": 20,


    "vcd_mode": "hybrid",  # or "pure_vcd" or "gradient_only"
    "vcd_num_views": 10,
    "vcd_error_threshold": 0.5,
    "vcd_densify_threshold": 6.0,
}




start_iteration = 0
latest_ckpt = None

if RESUME_FROM_CHECKPOINT:
    ckpt_dir = PROJECT_ROOT / "outputs" / "maria_bedroom" / training_run_n / "checkpoints"
    checkpoints = sorted(glob.glob(str(ckpt_dir / "*.pt")))
    print(checkpoints)

    if checkpoints:
        latest_ckpt = checkpoints[-1]
        print(f"Loading checkpoint: {latest_ckpt}")
        ckpt = torch.load(latest_ckpt, map_location="cuda")

        # Support both common formats
        if "model_state_dict" in ckpt:
            model.load_state_dict(ckpt["model_state_dict"])
            start_iteration = int(ckpt.get("iteration", 0))
        else:
            model.load_state_dict(ckpt)
            # infer from filename if needed
            stem = Path(latest_ckpt).stem  # checkpoint_010000
            try:
                start_iteration = int(stem.split("_")[-1])
            except Exception:
                start_iteration = 0

        print(f"Resumed from iteration {start_iteration}")
    else:
        print("No checkpoint found, starting fresh")

trainer = GaussianTrainerVCD(
    model=model,
    scene=scene_obj,
    scene_name="maria_bedroom",
    config=config,
    base_output_dir=str(PROJECT_ROOT / "outputs"),
)

#trainer.train()
trainer.export_best_marble(latest_ckpt)


['/workspace/Home_Reconstruction/outputs/maria_bedroom/training_run_9/checkpoints/ckpt_iter_009000.pt', '/workspace/Home_Reconstruction/outputs/maria_bedroom/training_run_9/checkpoints/ckpt_iter_010000.pt']
Loading checkpoint: /workspace/Home_Reconstruction/outputs/maria_bedroom/training_run_9/checkpoints/ckpt_iter_010000.pt


/tmp/ipykernel_589/1684081712.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(latest_ckpt, map_location="cuda")
07:27:35 | INFO    | ================

Resumed from iteration 10000


07:27:36 | INFO    | ======================================================================
07:27:36 | INFO    | TRAINING CONFIGURATION (VCD-ENHANCED)
07:27:36 | INFO    | ======================================================================
07:27:36 | INFO    |   adaptive_offset_cap_multiplier: 3.0
07:27:36 | INFO    |   checkpoint_metric: test_l1
07:27:36 | INFO    |   checkpoint_topk: 3
07:27:36 | INFO    |   densify_grad_threshold: 1e-05
07:27:36 | INFO    |   densify_interval: 500
07:27:36 | INFO    |   densify_start: 100
07:27:36 | INFO    |   densify_until: 9000
07:27:36 | INFO    |   early_stop_patience_evals: 20
07:27:36 | INFO    |   eval_interval: 1000
07:27:36 | INFO    |   lambda_offset_leash: 0.3
07:27:36 | INFO    |   lambda_scale_reg: 8.0
07:27:36 | INFO    |   lambda_semantic: 0.05
07:27:36 | INFO    |   lambda_ssim: 0.3
07:27:36 | INFO    |   lambda_volume: 0.0001
07:27:36 | INFO    |   lower_is_better: True
07:27:36 | INFO    |   lr: 0.001
07:27:36 | INFO    |   lr_

{'marble_scene_ply': PosixPath('/workspace/Home_Reconstruction/outputs/maria_bedroom/training_run_10/final_outputs/marble_iter_010000.ply'),
 'marble_objects_dir': PosixPath('/workspace/Home_Reconstruction/outputs/maria_bedroom/training_run_10/final_outputs/marble_objects')}

In [4]:
!nvidia-smi
# from time_training import profile

# # After creating trainer
# profile(trainer, num_iterations=100)

Mon Jan  5 04:36:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.65.06              Driver Version: 580.65.06      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:24:00.0 Off |                  Off |
|  0%   34C    P8             20W /  450W |       4MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
from time_training import profile

# After creating trainer
profile(trainer, num_iterations=100)

Warming up...
Profiling 100 iterations...

TIMING BREAKDOWN (averaged over 100 iterations)
backward            :   71.87 ±  0.60 ms ( 58.1%)  █████████████████████████████
get_params          :   18.66 ±  0.04 ms ( 15.1%)  ███████
render              :   15.20 ±  0.20 ms ( 12.3%)  ██████
semantic_loss       :   14.09 ±  0.04 ms ( 11.4%)  █████
optimizer_step      :    3.02 ±  0.14 ms (  2.4%)  █
rgb_loss            :    0.79 ±  0.11 ms (  0.6%)  
----------------------------------------------------------------------
TOTAL               :  123.64 ms  (8.1 it/s)

INSIGHTS:
  • Biggest bottleneck: backward (58.1%)
  • Forward: 48.7 ms (39.4%)
  • Backward: 74.9 ms (60.6%)


{'get_params': {'avg_ms': 18.65710536018014,
  'std_ms': 0.03999590431639459,
  'pct': 15.090309468372675},
 'render': {'avg_ms': 15.204297956079245,
  'std_ms': 0.20056797921567884,
  'pct': 12.29759691963098},
 'rgb_loss': {'avg_ms': 0.7928231731057167,
  'std_ms': 0.11278940415269305,
  'pct': 0.64125419270007},
 'semantic_loss': {'avg_ms': 14.088865262456238,
  'std_ms': 0.04084639253205782,
  'pct': 11.395408492596811},
 'backward': {'avg_ms': 71.87385343946517,
  'std_ms': 0.595013258626768,
  'pct': 58.13327791999557},
 'optimizer_step': {'avg_ms': 3.0193884391337633,
  'std_ms': 0.13748070210321617,
  'pct': 2.4421530067038915},
 '_total_ms': 123.63633363042027,
 '_iterations_per_sec': 8.088237257093441}

In [32]:
import torch
import gc



# del model
# del trainer

# Clear Python garbage
gc.collect()

# Clear CUDA cache
torch.cuda.empty_cache()

# Verify
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

print(torch.cuda.memory_summary(device='cuda:0', abbreviated=False))

import gc
import torch

tensors = []
for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            size_mb = obj.element_size() * obj.numel() / 1e6
            tensors.append((size_mb, obj.shape, obj.dtype))
    except:
        pass

print(f"Found {len(tensors)} CUDA tensors")
print(f"Total: {sum(t[0] for t in tensors):.1f} MB")
print()
print("Top 20 largest:")
print("-" * 70)
for size_mb, shape, dtype in tensors[:20]:
    print(f"{size_mb:>10.2f} MB  {str(shape):40s} {dtype}")


Allocated: 9.32 GB
Reserved: 12.39 GB
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 1            |        cudaMalloc retries: 3178      |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   8892 MiB |  22763 MiB | 147938 GiB | 147930 GiB |
|       from large pool |   8795 MiB |  22664 MiB | 147834 GiB | 147825 GiB |
|       from small pool |     96 MiB |    111 MiB |    104 GiB |    104 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   8892 MiB |  22763 MiB | 147938 GiB | 147930 GiB |
|       from large pool | 

In [13]:
from pathlib import Path

# ============================================================
# View individual object PLYs from training output
# ============================================================

# # Point to your training run's objects folder
# run_dir = Path("/workspace/Home_Reconstruction/outputs/maria bedroom/training_run_7/final_outputs/objects")

# # List available objects
# print("Available objects:")
# for ply in sorted(run_dir.glob("*.ply")):
#     print(f"  {ply.name}")

# # View a single object by ID
# object_id = 8  # e.g., 1 = nightstand
# ply_file = list(run_dir.glob(f"{object_id:02d}_*.ply"))[0]
# visualize_pointclouds(run_dir, [ply_file.name])



# Point to your training run's objects folder
splat_dir = Path("/workspace/Home_Reconstruction/outputs/maria_bedroom/training_run_3/final_outputs")
ply_name = "splat_supersplat.ply"
visualize_pointclouds(splat_dir, [ply_name])

SyntaxError: invalid syntax (1280567074.py, line 24)

In [ ]:
# ============================================================
# Run the pipeline
# ============================================================
# Comprehensive bedroom prompts
text_prompts = [
    # Main furniture
    "bed",
    "nightstand",
    "dresser",
    "desk",
    "chair",
    "wardrobe",
    "mirror",
    "clothing",
    "jacket",
    "shirt",
    
    # Lighting
    "lamp",
    "ceiling light",
    
    # Textiles
    "pillow",
    "blanket",
    "curtain",
    "rug",
    
    # Structure
    "wall",
    "floor",
    "ceiling",
    "door",
    "window",
    
    # Decor
    "plant",
    "picture frame",
    "book",
    "jewelry",
    "monitor",
    "laptop",
    "jacket",
    "towel",
    "paper",
    "computer",
    "shelf",
    "poster",
    "postcard",
]